# Introduction

This notebook continues the work from churn_data.ipynb, which handled the exploratory data analysis and the first preprocessing step. That step dropped 13 columns that either carried no predictive value (like CustomerID) or introduced data leakage (like Churn Score and Churn Reason), based on the exclusions set out in the proposal.

The output of that notebook, cleaned_churn_data.csv, is the starting point here. It still has the 11 blank Total Charges values and all the categorical columns as text, since fixing those now happens after the train and test split. This keeps the median used for imputation and the categories learned during encoding based only on the training data, avoiding the data leakage flagged during supervisor feedback.

This notebook covers the remaining preprocessing, class imbalance handling with SMOTE, training and tuning three models `(Logistic Regression, Random Forest, and LightGBM)`, evaluating them, and applying SHAP to the best performing model.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("cleaned_churn_data.csv")
print("Shape:", df.shape)
df.head()

Shape: (7043, 20)


,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,1
3,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,1
4,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,1


# Step 1: Convert Total Charges to a Number

Total Charges is currently stored as text, and the 11 missing values are blank spaces rather than true empty cells. This step converts the column to a numeric type, turning those blanks into proper missing values (NaN) so they can be handled correctly later. The values are not filled in yet, that happens after the train and test split, using only the training data.

In [2]:
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce")
print("Missing Total Charges after conversion:", df["Total Charges"].isna().sum())
df["Total Charges"].dtype

Missing Total Charges after conversion: 11


dtype('float64')

# Step 2: Train and Test Split

The dataset is split into training and test sets before any imputation, encoding, or scaling happens. This is the key step for avoiding data leakage, everything that comes after (filling in Total Charges, encoding categories, scaling, SMOTE) will be fit only on the training data, then applied to the test data using that same fit. The test set stays untouched until final evaluation.

A stratified 80/20 split is used, with a fixed random seed of 42 for reproducibility. Stratifying on Churn Value keeps the same churn rate (about 26.5%) in both the training and test sets, rather than letting one set end up with a disproportionate number of churners by chance.

In [3]:
X = df.drop(columns=["Churn Value"])
y = df["Churn Value"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print()
print("Churn rate in training set:", y_train.mean().round(4))
print("Churn rate in test set:", y_test.mean().round(4))

Training set shape: (5634, 19)
Test set shape: (1409, 19)

Churn rate in training set: 0.2654
Churn rate in test set: 0.2654


# Step 3: Preprocessing Pipeline

This step handles the missing Total Charges values, encodes the categorical columns, and scales the numeric columns, all in one pipeline. The pipeline is fit only on the training data, then used to transform both the training and test data. This means the median used to fill in Total Charges, the categories the encoders learn, and the scaling parameters are all worked out from training data alone, matching the approach used by Sana et al. (2022) for this dataset and avoiding data leakage.

The columns are grouped by how they need to be handled:

- Numeric columns (Tenure Months, Monthly Charges, Total Charges) get the missing values filled with the median, then get scaled using StandardScaler. Scaling matters most for Logistic Regression, since it is sensitive to features being on different numeric scales, but it is applied to all three numeric columns regardless of which model uses them.
- Binary columns (Gender, Senior Citizen, Partner, Dependents, Phone Service, Paperless Billing) only have two categories each, so they are label encoded into 0 and 1.
- Multi category columns (Multiple Lines, Internet Service, Online Security, Online Backup, Device Protection, Tech Support, Streaming TV, Streaming Movies, Contract, Payment Method) are one hot encoded, since they have three or more categories and label encoding would wrongly imply an order between them.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder

numeric_cols = ["Tenure Months", "Monthly Charges", "Total Charges"]
binary_cols = ["Gender", "Senior Citizen", "Partner", "Dependents", "Phone Service", "Paperless Billing"]
multi_cols = [
    "Multiple Lines", "Internet Service", "Online Security", "Online Backup",
    "Device Protection", "Tech Support", "Streaming TV", "Streaming Movies",
    "Contract", "Payment Method"
]

numeric_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

binary_pipeline = Pipeline(steps=[
    ("encode", OrdinalEncoder())
])

multi_pipeline = Pipeline(steps=[
    ("encode", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_cols),
    ("bin", binary_pipeline, binary_cols),
    ("multi", multi_pipeline, multi_cols)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
X_train_processed = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_test_processed = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

print("Training set shape after preprocessing:", X_train_processed.shape)
print("Test set shape after preprocessing:", X_test_processed.shape)
X_train_processed.head()

Training set shape after preprocessing: (5634, 30)
Test set shape after preprocessing: (1409, 30)


,num__Tenure Months,num__Monthly Charges,num__Total Charges,bin__Gender,bin__Senior Citizen,bin__Partner,bin__Dependents,bin__Phone Service,bin__Paperless Billing,multi__Multiple Lines_No phone service,...,multi__Tech Support_Yes,multi__Streaming TV_No internet service,multi__Streaming TV_Yes,multi__Streaming Movies_No internet service,multi__Streaming Movies_Yes,multi__Contract_One year,multi__Contract_Two year,multi__Payment Method_Credit card (automatic),multi__Payment Method_Electronic check,multi__Payment Method_Mailed check
4626,0.102371,-0.521976,-0.263290,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4192,-0.711743,0.337478,-0.504815,1.0,0.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
5457,-0.793155,-0.809013,-0.751214,1.0,0.0,1.0,1.0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
4717,-0.263980,0.284384,-0.173700,0.0,0.0,1.0,0.0,1.0,1.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0
4673,-1.281624,-0.676279,-0.990851,1.0,0.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


# Step 4: Handling Class Imbalance with SMOTE


The training set has a churn rate of about 26.5%, meaning churned customers are a minority. Left as is, this tends to bias models toward the majority class, giving misleadingly high accuracy while missing most actual churners, exactly the risk highlighted by Suguna et al. (2025).

SMOTE is applied here to fix that. It works by creating synthetic examples of the minority class based on existing churned customers, rather than just duplicating rows. Following Bilal et al. (2022), SMOTE is applied only to the training set, after the split and after encoding. The test set is left with its original, real world class distribution, since that is what the model needs to be evaluated against.

In [8]:
%pip install imbalanced-learn
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_processed, y_train)

print("Before SMOTE:")
print(y_train.value_counts())
print()
print("After SMOTE:")
print(y_train_resampled.value_counts())

  Obtaining dependency information for imbalanced-learn from https://files.pythonhosted.org/packages/91/54/760ccac7d8feeea0c191767a6b025d3ca5014084443ff48afb6fb24ef056/imbalanced_learn-0.14.2-py3-none-any.whl.metadata
  Obtaining dependency information for sklearn-compat<0.2,>=0.1.6 from https://files.pythonhosted.org/packages/b0/20/47a7e947757008be1b77f1c6a6861d26a84ef4b4ea3e6ebf4eff24f24d5d/sklearn_compat-0.1.6-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/236.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/236.1 kB ? eta -:--:--
   - -------------------------------------- 10.2/236.1 kB ? eta -:--:--
   ----- --------------------------------- 30.7/236.1 kB 435.7 kB/s eta 0:00:01
   ------ -------------------------------- 41.0/236.1 kB 326.8 kB/s eta 0:00:01
   ---------- ---------------------------- 61.4/236.1 kB 326.1 kB/s eta 0:00:01
   ------------- ------------------------- 81.9/236.1 kB 381.3 kB/s eta 0:00:01
   --------------- ----


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Before SMOTE:
Churn Value
0    4139
1    1495
Name: count, dtype: int64

After SMOTE:
Churn Value
0    4139
1    4139
Name: count, dtype: int64
